# 07. Deep Learning

> Compare the hand-crafted feature pipeline against an end-to-end CNN that operates on raw epochs: EEGNet is the canonical compact baseline for ECoG/EEG decoding.

Same stratified-k-fold protocol as the classical models in `06_classification` so results are directly comparable. Definitions execute in CI (cheap); the actual training/cross-validation cells are `#| eval: false` since they're slow.

In [ ]:
#| default_exp dl

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold

## EEGNet

Compact two-block CNN from Lawhern et al. (2018). Block 1 learns a temporal filter then a per-electrode spatial filter; Block 2 is a separable convolution. Tiny by deep-learning standards (~2k parameters), which is what makes it fit 90 trials without overfitting catastrophically.

In [ ]:
#| export
class EEGNet(nn.Module):
    def __init__(self, n_classes, n_channels, n_samples,
                 F1=8, D=2, F2=16, kernel_length=64, dropout=0.5):
        super().__init__()
        self.conv1     = nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False)
        self.bn1       = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2       = nn.BatchNorm2d(F1 * D)
        self.pool1     = nn.AvgPool2d((1, 4))
        self.drop1     = nn.Dropout(dropout)
        self.sepconv   = nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False)
        self.bn3       = nn.BatchNorm2d(F2)
        self.pool2     = nn.AvgPool2d((1, 8))
        self.drop2     = nn.Dropout(dropout)
        out_t          = n_samples // 4 // 8
        self.fc        = nn.Linear(F2 * out_t, n_classes)

    def forward(self, x):
        x = self.bn1(self.conv1(x))
        x = F.elu(self.bn2(self.depthwise(x)))
        x = self.drop1(self.pool1(x))
        x = F.elu(self.bn3(self.sepconv(x)))
        x = self.drop2(self.pool2(x))
        return self.fc(x.flatten(1))

## Training loop

Per-fold training: Adam, cross-entropy, fixed epoch count. Returns the trained model so the caller can run inference on a held-out fold.

In [ ]:
#| export
def train_eegnet(epochs, y, n_epochs=200, batch_size=16, lr=1e-3, device='cpu', **net_kwargs):
    """Train an EEGNet on `(trials, channels, samples)` epochs with integer labels `y`."""
    n_trials, n_channels, n_samples = epochs.shape
    n_classes = len(np.unique(y))
    X  = torch.from_numpy(epochs[:, None, :, :]).float().to(device)
    yt = torch.from_numpy(y).long().to(device)
    model = EEGNet(n_classes, n_channels, n_samples, **net_kwargs).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(n_epochs):
        idx = torch.randperm(n_trials, device=device)
        for i in range(0, n_trials, batch_size):
            b = idx[i:i + batch_size]
            opt.zero_grad()
            F.cross_entropy(model(X[b]), yt[b]).backward()
            opt.step()
    return model

## Cross-validation

Stratified k-fold mirroring `06_classification.cross_validate`. Labels are remapped to `0..n_classes-1` per fold for the cross-entropy loss.

In [ ]:
#| export
def cross_validate_eegnet(epochs, classes, n_splits=5, random_state=0, device='cpu', **train_kwargs):
    """Stratified k-fold accuracy for EEGNet on `(trials, channels, samples)` epochs."""
    label_to_idx = {c: i for i, c in enumerate(np.unique(classes))}
    y = np.array([label_to_idx[c] for c in classes])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs = []
    for tr, te in skf.split(epochs, y):
        model = train_eegnet(epochs[tr], y[tr], device=device, **train_kwargs)
        model.eval()
        with torch.no_grad():
            X_te = torch.from_numpy(epochs[te][:, None, :, :]).float().to(device)
            pred = model(X_te).argmax(1).cpu().numpy()
        accs.append((pred == y[te]).mean())
    return np.array(accs)

## Run

In [ ]:
#| eval: false
from br41n_ecog_hand_pose.data import load_ecog
from br41n_ecog_hand_pose.preprocessing import preprocess
from br41n_ecog_hand_pose.epoching import epoch_recording

rec = load_ecog()
clean, _ = preprocess(rec.ecog, rec.fs)
epochs, classes = epoch_recording(rec, tmin=0.0, tmax=2.0, signal=clean)
print(f'epochs {epochs.shape}, classes {classes.shape}')

In [ ]:
#| eval: false
accs = cross_validate_eegnet(epochs.astype('float32'), classes, n_splits=5, n_epochs=300, lr=1e-3)
print(f'EEGNet 5-fold accuracy: {accs.mean():.3f} \u00b1 {accs.std():.3f}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()